# Automatic entity type detection

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')

# Add the project root to the Python path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Import AAAIM functions
from core import annotate_model, curate_model
from core.database_search import force_clear_chromadb, get_species_recommendations_rag
from utils.evaluation import (
    evaluate_single_model,
    evaluate_models_in_folder,
    print_evaluation_results,
    compare_results,
    process_saved_llm_responses
)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# LLM configuration
# llm_model = "Llama-4-Maverick-17B-128E-Instruct-FP8"
llm_model = "Llama-3.3-70B-Instruct"
# llm_model = "meta-llama/llama-3.3-70b-instruct:free"
# llm_model = "gpt-4.1-nano"

output_dir = "./autoType/"  # Output directory for results

In [2]:
test_model_file = "/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels/BIOMD0000000052.xml"
# test_model_file = "190_few_anno.xml"
# Check if test model exists
if os.path.exists(test_model_file):
    print(f"✓ Test model found: {test_model_file}")
else:
    print(f"✗ Test model not found: {test_model_file}")

✓ Test model found: /Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels/BIOMD0000000052.xml


In [3]:
from core.model_info import format_prompt, find_species_with_annotations_and_qualifiers
# extract species that have chebi annotation
existing_annotations, qualifiers = find_species_with_annotations_and_qualifiers(test_model_file, 'chebi')
existing_annotations

{'Glu': ['17234'],
 'Fru': ['28757'],
 'Formic_acid': ['30751'],
 'Acetic_acid': ['15366'],
 'lys_R': ['32568']}

In [4]:
# get the prompt for this model
prompt = format_prompt(test_model_file, existing_annotations.keys(), 'chemical')
print(prompt)

Now annotate these:
Chemical to annotate: Glu, Fru, Formic_acid, Acetic_acid, lys_R
Model: "Brands2002 - Monosaccharide-casein systems"

// Reactions:
Glu => Fru
Fru => Glu
Glu => C5 + Formic_acid
Fru => C5 + Formic_acid
Fru => 2 Triose
Triose => Cn + Acetic_acid
lys_R + Glu => Amadori
Amadori => Acetic_acid + lys_R
lys_R + Fru => AMP

// Notes:
"Brands2002 - Monosaccharide-casein systems
A kinetic model of the Maillard reaction occurring in heated monosaccharide-casein system.
This model is described in the article:
Kinetic modeling of reactions in heated monosaccharide-casein systems.
Brands CM, van Boekel MA
Journal of Agricultural and Food Chemistry. 2002, 50(23):6725-6739
Abstract:
In the present study, a kinetic model of the Maillard reaction occurring in heated monosaccharide-casein systems was proposed. Its parameters, the reaction rate constants, were estimated via multiresponse modeling. The determinant criterion was used as the statistical fit criterion instead of the famili

In [5]:
result_df = evaluate_single_model(
    model_file=test_model_file,
    llm_model=llm_model,
    method = 'direct',
    top_k = 5,
    entity_type='chemical',
    database='chebi',
    save_llm_results=False,
    output_dir=output_dir,
    verbose=True
)

2025-11-24 13:13:33,786 - INFO - Evaluating model: BIOMD0000000023.xml
2025-11-24 13:13:33,889 - INFO - Evaluating 13 entities in BIOMD0000000023.xml
2025-11-24 13:13:37,702 - INFO - HTTP Request: POST https://api.llama.com/compat/v1/chat/completions "HTTP/1.1 200 OK"


LLM response: 
Fru: "fructose", "D-fructose", "beta-D-fructose"
Glc: "glucose", "D-glucose", "alpha-D-glucose"
HexP: "hexose phosphate", "fructose-6-phosphate", "glucose-6-phosphate"
Suc6P: "sucrose-6-phosphate", "sucrose phosphate", "6-phosphosucrose"
Suc: "sucrose", "alpha-D-glucopyranosyl beta-D-fructofuranoside", "beta-D-fructofuranosyl alpha-D-glucopyranoside"
Sucvac: "sucrose", "vacuolar sucrose", "sucrose in vacuole"
Glcex: "glucose", "extracellular glucose", "external glucose"
Fruex: "fructose", "extracellular fructose", "external fructose"
phos: "phosphate", "inorganic phosphate", "orthophosphate"
UDP: "uridine diphosphate", "uridine 5'-diphosphate", "UDP"
ADP: "adenosine diphosphate", "adenosine 5'-diphosphate", "ADP"
ATP: "adenosine triphosphate", "adenosine 5'-triphosphate", "ATP"
glycolysis: "glycolytic pathway", "glycolysis intermediate", "UNK"
Reason: The model appears to be a metabolic model of sucrose metabolism, with reactions involving glucose, fructose, and sucrose.

In [6]:
result_df

,model,species_id,display_name,synonyms_LLM,reason,exist_annotation_id,exist_annotation_name,predictions,predictions_names,match_score,...,recall_exact,precision_exact,accuracy,total_time,llm_time,query_time,tax_id,tax_name,qualifier,detected_entity_type
0,BIOMD0000000023.xml,Fru,,"[fructose, D-fructose, beta-D-fructose]",Chunk 1: The model appears to be a metabolic m...,[15824],D-fructose,"[28645, 28757, 15824, 37721, 48095]","beta-D-fructofuranose, fructose, D-fructose, D...","[0.6666666666666666, 0.3333333333333333, 0.333...",...,1.00,0.20,1,5.373753,3.784719,1.589034,None,None,is,unknown
1,BIOMD0000000023.xml,Glc,,"[glucose, D-glucose, alpha-D-glucose]",,[17634],D-glucose,"[4167, 42758, 17234, 17634, 17925]","D-glucopyranose, aldehydo-D-glucose, glucose, ...","[0.6666666666666666, 0.6666666666666666, 0.333...",...,1.00,0.20,1,5.373753,3.784719,1.589034,None,None,is,unknown
2,BIOMD0000000023.xml,HexP,,"[hexose phosphate, fructose-6-phosphate, gluco...",,"[16218, 15946, 14314, 18066]","beta-D-glucose 1-phosphate, keto-D-fructose 6-...","[47878, 16084, 15946, 88003]","hexose phosphate, beta-D-fructofuranose 6-phos...","[0.3333333333333333, 0.3333333333333333, 0.333...",...,0.25,0.25,1,5.373753,3.784719,1.589034,None,None,"hasVersion, hasVersion, hasVersion, hasVersion",unknown
3,BIOMD0000000023.xml,Suc6P,,"[sucrose-6-phosphate, sucrose phosphate, 6-pho...",,[16308],sucrose 6(F)-phosphate,"[131603, 16308]","sucrose 6(G)-phosphate, sucrose 6(F)-phosphate","[0.6666666666666666, 0.6666666666666666]",...,1.00,0.50,1,5.373753,3.784719,1.589034,None,None,is,unknown
4,BIOMD0000000023.xml,Suc,,"[sucrose, alpha-D-glucopyranosyl beta-D-fructo...",,[17992],sucrose,[17992],sucrose,[0.6666666666666666],...,1.00,1.00,1,5.373753,3.784719,1.589034,None,None,is,unknown
5,BIOMD0000000023.xml,Sucvac,,"[sucrose, vacuolar sucrose, sucrose in vacuole]",,[17992],sucrose,[17992],sucrose,[0.3333333333333333],...,1.00,1.00,1,5.373753,3.784719,1.589034,None,None,is,unknown
6,BIOMD0000000023.xml,glycolysis,,"[glycolytic pathway, glycolysis intermediate, ...",,[28013],"beta-D-fructofuranose 1,6-bisphosphate",[],NA,[],...,0.00,0.00,0,5.373753,3.784719,1.589034,None,None,is,unknown
7,BIOMD0000000023.xml,phos,,"[phosphate, inorganic phosphate, orthophosphate]",,[18367],phosphate(3-),"[18367, 26020, 26078, 35780, 43474]","phosphate(3-), phosphate, phosphoric acid, pho...","[0.6666666666666666, 0.3333333333333333, 0.333...",...,1.00,0.20,1,5.373753,3.784719,1.589034,None,None,is,unknown
8,BIOMD0000000023.xml,UDP,,"[uridine diphosphate, uridine 5'-diphosphate, ...",,[17659],UDP,"[17659, 58223]","UDP, UDP(3-)","[1.0, 0.6666666666666666]",...,1.00,0.50,1,5.373753,3.784719,1.589034,None,None,is,unknown
9,BIOMD0000000023.xml,ADP,,"[adenosine diphosphate, adenosine 5'-diphospha...",,[16761],ADP,"[16761, 456216, 177051, 73342]","ADP, ADP(3-), Adenosine_Diphosphate, Ala-Asp-Pro","[0.6666666666666666, 0.6666666666666666, 0.333...",...,1.00,0.25,1,5.373753,3.784719,1.589034,None,None,is,unknown


# Test on auto detection

In [5]:
import sys
import importlib
importlib.reload(sys.modules['core.model_info'])
importlib.reload(sys.modules['utils.evaluation'])
from core.model_info import format_prompt, find_species_with_annotations_and_qualifiers
from utils.evaluation import evaluate_single_model

In [6]:
prompt = format_prompt(test_model_file, existing_annotations.keys(), 'auto')
print(prompt)

Now annotate these species:
Species to annotate: Glu, Fru, Formic_acid, Acetic_acid, lys_R
Model: "Brands2002 - Monosaccharide-casein systems"

// Reactions:
Glu => Fru
Fru => Glu
Glu => C5 + Formic_acid
Fru => C5 + Formic_acid
Fru => 2 Triose
Triose => Cn + Acetic_acid
lys_R + Glu => Amadori
Amadori => Acetic_acid + lys_R
lys_R + Fru => AMP

// Notes:
"Brands2002 - Monosaccharide-casein systems
A kinetic model of the Maillard reaction occurring in heated monosaccharide-casein system.
This model is described in the article:
Kinetic modeling of reactions in heated monosaccharide-casein systems.
Brands CM, van Boekel MA
Journal of Agricultural and Food Chemistry. 2002, 50(23):6725-6739
Abstract:
In the present study, a kinetic model of the Maillard reaction occurring in heated monosaccharide-casein systems was proposed. Its parameters, the reaction rate constants, were estimated via multiresponse modeling. The determinant criterion was used as the statistical fit criterion instead of the

In [7]:
result_df = evaluate_single_model(
    model_file=test_model_file,
    llm_model=llm_model,
    method = 'direct',
    top_k = 3,
    entity_type='auto',
    database='chebi',
    save_llm_results=False,
    output_dir=output_dir,
    verbose=True
)

2025-11-26 23:06:29,914 - INFO - Evaluating model: BIOMD0000000052.xml
2025-11-26 23:06:29,981 - INFO - Evaluating 4 entities in BIOMD0000000052.xml
2025-11-26 23:06:32,136 - INFO - HTTP Request: POST https://api.llama.com/compat/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-26 23:06:32,150 - INFO - Detected entity types: {'chemical': 3, 'protein': 1}
2025-11-26 23:06:32,150 - INFO - Searching chebi for 3 chemical entities
2025-11-26 23:06:33,071 - WARNING - No valid database found for entity type 'protein' in allowed databases: ['chebi']
2025-11-26 23:06:33,071 - WARNING - No valid database found for entity type 'protein' in ['chebi'] for 1 species


In [8]:
result_df

,model,species_id,display_name,detected_entity_type,synonyms_LLM,reason,exist_annotation_id,exist_annotation_name,predictions,predictions_names,...,precision_formula,recall_exact,precision_exact,accuracy,total_time,llm_time,query_time,tax_id,tax_name,qualifier
0,BIOMD0000000052.xml,Glu,,chemical,"[glucose, D-glucose, blood sugar]",Chunk 1: The model describes the Maillard reac...,[17234],glucose,"[4167, 42758, 17234]","D-glucopyranose, aldehydo-D-glucose, glucose",...,1.0,1.0,0.333,1,3.073333,2.15174,0.921593,None,None,is
1,BIOMD0000000052.xml,Formic_acid,,chemical,"[formic acid, methanoic acid, HCOOH]",,[30751],formic acid,[30751],formic acid,...,1.0,1.0,1.000,1,3.073333,2.15174,0.921593,None,None,is
2,BIOMD0000000052.xml,Acetic_acid,,chemical,"[acetic acid, ethanoic acid, CH3COOH]",,[15366],acetic acid,[15366],acetic acid,...,1.0,1.0,1.000,1,3.073333,2.15174,0.921593,None,None,is
3,BIOMD0000000052.xml,lys_R,,protein,"[lysine receptor, lysine, L-lysine]",,[32568],lysine residue,[],NA,...,0.0,0.0,0.000,0,3.073333,2.15174,0.921593,None,None,is


In [8]:
from core.llm_interface import SYSTEM_PROMPT, query_llm, parse_llm_response, get_system_prompt
llm_response = """
Ca_cyt (chemical): "calcium(2+)", "calcium ion", "Ca2+"
CaER (chemical): "calcium", "endoplasmic reticulum calcium", "Ca2+ in endoplasmic reticulum"
CaM (chemical): "calmodulin", "calcium-modulated protein", "CaM protein"
CaPr (complex): "calcium-protein complex", "protein-calcium complex", "calcium bound protein"

Reason: The model "Marhl2000_CaOscillations" suggests a focus on calcium oscillations
"""
chunk_synonyms_dict, chunk_entity_type_dict, chunk_reason = parse_llm_response(llm_response)

In [29]:
# Test with a single model using automatic entity type detection
recommendations_df, metrics = annotate_model(
    model_file=test_model_file,
    llm_model=llm_model,
    method="direct",
    top_k=3,
    entity_type="auto",
    database=["chebi", "uniprot"]
)

2025-11-17 14:57:08,492 - INFO - Starting annotation for model: /Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels/BIOMD0000000039.xml
2025-11-17 14:57:08,493 - INFO - Using LLM model: Llama-3.3-70B-Instruct
2025-11-17 14:57:08,493 - INFO - Using method: direct for database search
2025-11-17 14:57:08,494 - INFO - Entity type: auto, Database: ['chebi', 'uniprot']
2025-11-17 14:57:08,494 - INFO - >>>Step 1: Getting species from model...<<<
2025-11-17 14:57:08,500 - INFO - Found 5 species in model
2025-11-17 14:57:08,501 - WARNING - Entity type auto with database ['chebi', 'uniprot'] not yet supported
2025-11-17 14:57:08,501 - INFO - Annotate all 5 entities
2025-11-17 14:57:08,501 - INFO - >>>Step 2: Extracting model context...<<<
2025-11-17 14:57:08,526 - INFO - Extracted context for model: Marhl2000_CaOscillations
2025-11-17 14:57:08,526 - INFO - >>>Step 3: Querying LLM (Llama-3.3-70B-Instruct)...<<<
2025-11-17 14:57:08,546 - WARNING - Auto entity type detection not yet implemented
202

## uniprot

In [ ]:
test_model_file = "/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels_251106/BIOMD0000000019.xml"
tax_id = 9606
# Check if test model exist
if os.path.exists(test_model_file):
    print(f"✓ Test model found: {test_model_file}")
else:
    print(f"✗ Test model not found: {test_model_file}")

✓ Test model found: /Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels_251106/BIOMD0000000019.xml


In [ ]:
# Running with given entity type
result_df = evaluate_single_model(
    model_file=test_model_file,
    tax_id=tax_id,
    llm_model=llm_model,
    entity_type='protein',
    database='uniprot',
    save_llm_results=False,
    output_dir=output_dir
)

2025-11-28 17:28:01,114 - INFO - Evaluating model: BIOMD0000000019.xml
2025-11-28 17:28:01,115 - INFO - Using organism-specific search for tax_id: 9606
2025-11-28 17:28:01,162 - INFO - Evaluating 17 entities in BIOMD0000000019.xml
2025-11-28 17:28:07,679 - INFO - HTTP Request: POST https://api.llama.com/compat/v1/chat/completions "HTTP/1.1 200 OK"


In [16]:
result_df

,model,species_id,display_name,detected_entity_type,synonyms_LLM,reason,exist_annotation_id,exist_annotation_name,predictions,predictions_names,...,precision_formula,recall_exact,precision_exact,accuracy,total_time,llm_time,query_time,tax_id,tax_name,qualifier
0,BIOMD0000000019.xml,x3,EGF-EGFR,protein,"[EGFR, ERBB1, HER1]",Chunk 1: The model appears to be an EGF MAPK s...,"[P01133, P00533]","EGF, EGFR",[P00533],EGFR,...,0,0.50,1.0,1,6.857546,6.358645,0.498901,9606,None,"hasVersion, hasVersion"
1,BIOMD0000000019.xml,x4,EGF-EGFR^2,protein,"[EGFR, ERBB1, HER1]",,"[P01133, P00533]","EGF, EGFR",[P00533],EGFR,...,0,0.50,1.0,1,6.857546,6.358645,0.498901,9606,None,"hasVersion, hasVersion"
2,BIOMD0000000019.xml,x5,EGF-EGFR*^2,protein,"[EGFR, ERBB1, HER1]",,"[P00533, P01133]","EGFR, EGF",[P00533],EGFR,...,0,0.50,1.0,1,6.857546,6.358645,0.498901,9606,None,"hasVersion, hasVersion"
3,BIOMD0000000019.xml,x6,EGFRi,protein,"[EGFR, ERBB1, HER1]",,[P00533],EGFR,[P00533],EGFR,...,0,1.00,1.0,1,6.857546,6.358645,0.498901,9606,None,isVersionOf
4,BIOMD0000000019.xml,x7,EGF-EGFR*^2-GAP-Grb2-Prot,protein,"[GRB2, GROWTH FACTOR RECEPTOR-BOUND PROTEIN 2,...",,"[P20936, P62993, P00533, P01133]","RASA1, GRB2, EGFR, EGF",[P62993],GRB2,...,0,0.25,1.0,1,6.857546,6.358645,0.498901,9606,None,"hasVersion, hasVersion, hasVersion, hasVersion"
5,BIOMD0000000019.xml,x8,EGF-EGFRi*^2,protein,"[EGFR, ERBB1, HER1]",,"[P00533, P01133]","EGFR, EGF",[P00533],EGFR,...,0,0.50,1.0,1,6.857546,6.358645,0.498901,9606,None,"hasVersion, hasVersion"
6,BIOMD0000000019.xml,x10,EGF-EGFRi,protein,"[EGFR, ERBB1, HER1]",,"[P01133, P00533]","EGF, EGFR",[P00533],EGFR,...,0,0.50,1.0,1,6.857546,6.358645,0.498901,9606,None,"hasVersion, hasVersion"
7,BIOMD0000000019.xml,x14,GAP,protein,"[GAP, GTPASE-ACTIVATING PROTEIN, RAS GTPASE-AC...",,[P20936],RASA1,[P20936],RASA1,...,0,1.00,1.0,1,6.857546,6.358645,0.498901,9606,None,isVersionOf
8,BIOMD0000000019.xml,x22,Grb2,protein,"[GRB2, GROWTH FACTOR RECEPTOR-BOUND PROTEIN 2,...",,[P62993],GRB2,[P62993],GRB2,...,0,1.00,1.0,1,6.857546,6.358645,0.498901,9606,None,isVersionOf
9,BIOMD0000000019.xml,x24,Sos,protein,"[SOS1, SON OF SEVENLESS, GROWTH FACTOR RECEPTO...",,[Q07889],SOS1,[Q07889],SOS1,...,0,1.00,1.0,1,6.857546,6.358645,0.498901,9606,None,isVersionOf


In [17]:
# Running with auto entity type
result_df = evaluate_single_model(
    model_file=test_model_file,
    tax_id=tax_id,
    llm_model=llm_model,
    entity_type='auto',
    database='uniprot',
    save_llm_results=False,
    output_dir=output_dir
)

2025-11-28 17:28:46,121 - INFO - Evaluating model: BIOMD0000000019.xml
2025-11-28 17:28:46,122 - INFO - Using organism-specific search for tax_id: 9606
2025-11-28 17:28:46,160 - INFO - Evaluating 17 entities in BIOMD0000000019.xml
2025-11-28 17:28:52,267 - INFO - HTTP Request: POST https://api.llama.com/compat/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-28 17:28:52,268 - INFO - Detected entity types: {'complex': 6, 'protein': 11}
2025-11-28 17:28:52,268 - INFO - Searching uniprot for 6 complex entities
2025-11-28 17:28:52,370 - INFO - Searching uniprot for 11 protein entities


In [18]:
result_df

,model,species_id,display_name,detected_entity_type,synonyms_LLM,reason,exist_annotation_id,exist_annotation_name,predictions,predictions_names,...,precision_formula,recall_exact,precision_exact,accuracy,total_time,llm_time,query_time,tax_id,tax_name,qualifier
0,BIOMD0000000019.xml,x3,EGF-EGFR,complex,"[EGF-EGFR, Epidermal Growth Factor Receptor co...",Chunk 1: The entity types were determined base...,"[P01133, P00533]","EGF, EGFR",[],NA,...,0,0.0,0.0,0,6.216499,5.964367,0.252132,9606,None,"hasVersion, hasVersion"
1,BIOMD0000000019.xml,x4,EGF-EGFR^2,complex,"[EGF-EGFR dimer, EGFR dimer, EGF-EGFR^2]",,"[P01133, P00533]","EGF, EGFR",[],NA,...,0,0.0,0.0,0,6.216499,5.964367,0.252132,9606,None,"hasVersion, hasVersion"
2,BIOMD0000000019.xml,x5,EGF-EGFR*^2,complex,"[EGF-EGFR^2-GAP, EGFR-GAP complex, GAP-EGFR co...",,"[P00533, P01133]","EGFR, EGF",[],NA,...,0,0.0,0.0,0,6.216499,5.964367,0.252132,9606,None,"hasVersion, hasVersion"
3,BIOMD0000000019.xml,x7,EGF-EGFR*^2-GAP-Grb2-Prot,complex,"[EGF-EGFR*^2-GAP-Grb2-Prot, EGFR-Grb2-GAP comp...",,"[P20936, P62993, P00533, P01133]","RASA1, GRB2, EGFR, EGF",[],NA,...,0,0.0,0.0,0,6.216499,5.964367,0.252132,9606,None,"hasVersion, hasVersion, hasVersion, hasVersion"
4,BIOMD0000000019.xml,x8,EGF-EGFRi*^2,complex,"[EGF-EGFRi*^2, Inactive EGFR dimer, EGFRi dimer]",,"[P00533, P01133]","EGFR, EGF",[],NA,...,0,0.0,0.0,0,6.216499,5.964367,0.252132,9606,None,"hasVersion, hasVersion"
5,BIOMD0000000019.xml,x10,EGF-EGFRi,complex,"[EGF-EGFRi, EGFRi, Inactive EGFR]",,"[P01133, P00533]","EGF, EGFR",[],NA,...,0,0.0,0.0,0,6.216499,5.964367,0.252132,9606,None,"hasVersion, hasVersion"
6,BIOMD0000000019.xml,x6,EGFRi,protein,"[EGFRi, Inactive EGFR, EGFR inhibitor]",,[P00533],EGFR,[],NA,...,0,0.0,0.0,0,6.216499,5.964367,0.252132,9606,None,isVersionOf
7,BIOMD0000000019.xml,x14,GAP,protein,"[GAP, GTPase-activating protein, RasGAP]",,[P20936],RASA1,"[P20936, Q15283]","RASA1, RASA2",...,0,1.0,0.5,1,6.216499,5.964367,0.252132,9606,None,isVersionOf
8,BIOMD0000000019.xml,x22,Grb2,protein,"[Grb2, Growth factor receptor-bound protein 2,...",,[P62993],GRB2,[P62993],GRB2,...,0,1.0,1.0,1,6.216499,5.964367,0.252132,9606,None,isVersionOf
9,BIOMD0000000019.xml,x24,Sos,protein,"[Sos, Son of sevenless, Guanine nucleotide exc...",,[Q07889],SOS1,[],NA,...,0,0.0,0.0,0,6.216499,5.964367,0.252132,9606,None,isVersionOf


# Batch evaluation
## chebi

In [2]:
model_dir = "/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels_251106"
# model_dir = "test_models"
# Check if model directory exists
if os.path.exists(model_dir):
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.xml')]
    print(f"✓ Model directory found: {model_dir}")
    print(f"  - Found {len(model_files)} XML files")
    # print(f"  - Will test first {min(num_models_to_test, len(model_files))} models")

✓ Model directory found: /Users/luna/Desktop/CRBM/AMAS_proj/Models/BioModels_251106
  - Found 1075 XML files


In [ ]:
# Run batch evaluation on updated BioModels
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    entity_type='auto',
    database='chebi',
    method="rag",
    top_k = 3,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts.csv",
    start_at=1
)

In [8]:
# Run batch evaluation 
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    entity_type='auto',
    database='chebi',
    method="rag",
    top_k = 10,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd251106_chebi_rag_llama-4_top10_autoType_updated_prompts_updated.csv",
    start_at=1
)

2025-12-06 14:20:19,157 - WARNING - Skipping BIOMD0000000001.xml - no results generated


LLM results will be saved to: ./autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_1420
Saved configuration to ./autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_1420/config.txt
Evaluating 1/1075: BIOMD0000000001.xml
Evaluating 2/1075: BIOMD0000000002.xml


2025-12-06 14:20:21,227 - WARNING - Skipping BIOMD0000000003.xml - no results generated
2025-12-06 14:20:21,231 - WARNING - Skipping BIOMD0000000004.xml - no results generated
2025-12-06 14:20:21,237 - WARNING - Skipping BIOMD0000000005.xml - no results generated
2025-12-06 14:20:21,240 - WARNING - Skipping BIOMD0000000006.xml - no results generated
2025-12-06 14:20:21,251 - WARNING - Skipping BIOMD0000000007.xml - no results generated
2025-12-06 14:20:21,256 - WARNING - Skipping BIOMD0000000008.xml - no results generated
2025-12-06 14:20:21,268 - WARNING - Skipping BIOMD0000000009.xml - no results generated
2025-12-06 14:20:21,275 - WARNING - Skipping BIOMD0000000010.xml - no results generated
2025-12-06 14:20:21,290 - WARNING - Skipping BIOMD0000000011.xml - no results generated
2025-12-06 14:20:21,299 - WARNING - Skipping BIOMD0000000012.xml - no results generated


LLM results saved to: autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_1420/BIOMD0000000002.txt
Evaluating 3/1075: BIOMD0000000003.xml
Evaluating 4/1075: BIOMD0000000004.xml
Evaluating 5/1075: BIOMD0000000005.xml
Evaluating 6/1075: BIOMD0000000006.xml
Evaluating 7/1075: BIOMD0000000007.xml
Evaluating 8/1075: BIOMD0000000008.xml
Evaluating 9/1075: BIOMD0000000009.xml
Evaluating 10/1075: BIOMD0000000010.xml
Evaluating 11/1075: BIOMD0000000011.xml
Evaluating 12/1075: BIOMD0000000012.xml
Evaluating 13/1075: BIOMD0000000013.xml


2025-12-06 14:20:30,130 - WARNING - Skipping BIOMD0000000014.xml - no results generated


...

Evaluating 1069/1075: BIOMD0000001095.xml
Evaluating 1070/1075: BIOMD0000001096.xml


2025-12-06 16:35:45,655 - WARNING - Skipping BIOMD0000001097.xml - no results generated
2025-12-06 16:35:45,816 - WARNING - Skipping BIOMD0000001098.xml - no results generated


Evaluating 1071/1075: BIOMD0000001097.xml
Evaluating 1072/1075: BIOMD0000001098.xml
Evaluating 1073/1075: BIOMD0000001099.xml


2025-12-06 16:35:45,993 - WARNING - Skipping BIOMD0000001099.xml - no results generated
2025-12-06 16:35:46,003 - WARNING - Skipping BIOMD0000001102.xml - no results generated
2025-12-06 16:35:46,011 - WARNING - Skipping BIOMD0000001103.xml - no results generated


Evaluating 1074/1075: BIOMD0000001102.xml
Evaluating 1075/1075: BIOMD0000001103.xml


In [3]:
# Run batch evaluation on updated BioModels
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model=llm_model,
    entity_type='auto',
    database='chebi',
    method="rag",
    top_k = 10,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd251106_chebi_rag_llama-3_top10_autoType.csv",
    start_at=1
)

2025-11-20 12:19:05,166 - WARNING - Skipping BIOMD0000000001.xml - no results generated


LLM results will be saved to: ./autoType/Llama-3.3-70B-instruct-Meta/auto/20251120_1219
Evaluating 1/1075: BIOMD0000000001.xml
Evaluating 2/1075: BIOMD0000000002.xml


2025-11-20 12:19:10,592 - WARNING - Skipping BIOMD0000000003.xml - no results generated
2025-11-20 12:19:10,598 - WARNING - Skipping BIOMD0000000004.xml - no results generated
2025-11-20 12:19:10,604 - WARNING - Skipping BIOMD0000000005.xml - no results generated
2025-11-20 12:19:10,608 - WARNING - Skipping BIOMD0000000006.xml - no results generated
2025-11-20 12:19:10,649 - WARNING - Skipping BIOMD0000000007.xml - no results generated
2025-11-20 12:19:10,666 - WARNING - Skipping BIOMD0000000008.xml - no results generated
2025-11-20 12:19:10,738 - WARNING - Skipping BIOMD0000000009.xml - no results generated
2025-11-20 12:19:10,757 - WARNING - Skipping BIOMD0000000010.xml - no results generated
2025-11-20 12:19:10,775 - WARNING - Skipping BIOMD0000000011.xml - no results generated


LLM results saved to: autoType/Llama-3.3-70B-instruct-Meta/auto/20251120_1219/BIOMD0000000002.txt
LLM response: 
L (chemical): "acetylcholine", "ACh", "achetylcholine"
Reason: The species L is identified as "ACh" in the display names, which is a common abbreviation for acetylcholine, a neurotransmitter. The model is also described as an "EPSP ACh species" and discusses nicotinic acetylcholine receptors, further supporting the identification of L as acetylcholine.
Synonyms dict: {'L': ['acetylcholine', 'ACh', 'achetylcholine']}
Evaluating 3/1075: BIOMD0000000003.xml
Evaluating 4/1075: BIOMD0000000004.xml
Evaluating 5/1075: BIOMD0000000005.xml
Evaluating 6/1075: BIOMD0000000006.xml
Evaluating 7/1075: BIOMD0000000007.xml
Evaluating 8/1075: BIOMD0000000008.xml
Evaluating 9/1075: BIOMD0000000009.xml
Evaluating 10/1075: BIOMD0000000010.xml
Evaluating 11/1075: BIOMD0000000011.xml
Evaluating 12/1075: BIOMD0000000012.xml


2025-11-20 12:19:10,785 - WARNING - Skipping BIOMD0000000012.xml - no results generated


...

Evaluating 1069/1075: BIOMD0000001095.xml
Evaluating 1070/1075: BIOMD0000001096.xml


2025-11-20 14:54:26,351 - WARNING - Skipping BIOMD0000001096.xml - no results generated
2025-11-20 14:54:26,603 - WARNING - Skipping BIOMD0000001097.xml - no results generated
2025-11-20 14:54:26,778 - WARNING - Skipping BIOMD0000001098.xml - no results generated


Evaluating 1071/1075: BIOMD0000001097.xml
Evaluating 1072/1075: BIOMD0000001098.xml
Evaluating 1073/1075: BIOMD0000001099.xml


2025-11-20 14:54:26,966 - WARNING - Skipping BIOMD0000001099.xml - no results generated
2025-11-20 14:54:26,975 - WARNING - Skipping BIOMD0000001102.xml - no results generated
2025-11-20 14:54:26,988 - WARNING - Skipping BIOMD0000001103.xml - no results generated


Evaluating 1074/1075: BIOMD0000001102.xml
Evaluating 1075/1075: BIOMD0000001103.xml


In [5]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-3_top10_autoType.csv")

Filtered results to 9312 entries that exist in reference: /Users/luna/Desktop/CRBM/AMAS_proj/Results/biomd_species_accuracy_AMAS.csv
Number of models assessed: 303
Number of models with predictions: 303
Number of annotations evaluated: 9312
Average accuracy (per model): 0.94
Ave. recall (formula): 0.94
Ave. precision (formula): 0.23
Ave. recall (exact): 0.88
Ave. precision (exact): 0.09
Average accuracy (per species): 0.94
Ave. recall (formula, per species): 0.94
Ave. precision (formula, per species): 0.22
Ave. recall (exact, per species): 0.74
Ave. precision (exact, per species): 0.11
Ave. total time (per model): 21.80
Ave. total time (per element, per model): 0.71
Ave. LLM time (per model): 21.00
Ave. LLM time (per element, per model): 0.68
Average number of predictions per species: 9.94


## Uniprot

In [ ]:
# Run batch evaluation on updated BioModels
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    entity_type='auto',
    database='uniprot',
    method="direct",
    top_k = 3,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd251106_uniprot_direct_llama-4_top3_autoType_updated_prompts.csv",
    start_at=1
)

In [13]:
print_evaluation_results("autoType/biomd251106_uniprot_direct_llama-4_top3_autoType.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 472
Number of models with predictions: 352
Number of annotations evaluated: 7825
Average accuracy (per model): 0.07
Ave. recall (formula): 0.00
Ave. precision (formula): 0.00
Ave. recall (exact): 0.07
Ave. precision (exact): 0.06
Average accuracy (per species): 0.03
Ave. recall (formula, per species): 0.00
Ave. precision (formula, per species): 0.00
Ave. recall (exact, per species): 0.03
Ave. precision (exact, per species): 0.02
Ave. total time (per model): 11.71
Ave. total time (per element, per model): 0.71
Ave. LLM time (per model): 11.51
Ave. LLM time (per element, per model): 0.69
Average number of predictions per species: 0.32


In [19]:
model_dir = '/Users/luna/Desktop/CRBM/AMAS_proj/Models/BioDivine/'

# Check if model directory exists
if os.path.exists(model_dir):
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.sbml')]
    print(f"✓ Model directory found: {model_dir}")
    print(f"  - Found {len(model_files)} SBML files")

✓ Model directory found: /Users/luna/Desktop/CRBM/AMAS_proj/Models/BioDivine/
  - Found 190 SBML files


In [ ]:
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    top_k=3,
    entity_type='auto',
    database='uniprot',
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biodivine_uniprot_direct_llama-3_top5_9606_autoType.csv",
    start_at=1,
    tax_id = 9606
)

In [22]:
print_evaluation_results("autoType/biodivine_uniprot_direct_llama-3_top5_9606_autoType.csv", ref_results_csv=None)

Showing all results
Number of models assessed: 15
Number of models with predictions: 8
Number of annotations evaluated: 605
Average accuracy (per model): 0.00
Ave. recall (formula): 0.00
Ave. precision (formula): 0.00
Ave. recall (exact): 0.00
Ave. precision (exact): 0.00
Average accuracy (per species): 0.00
Ave. recall (formula, per species): 0.00
Ave. precision (formula, per species): 0.00
Ave. recall (exact, per species): 0.00
Ave. precision (exact, per species): 0.00
Ave. total time (per model): 14.54
Ave. total time (per element, per model): 0.36
Ave. LLM time (per model): 14.34
Ave. LLM time (per element, per model): 0.36
Average number of predictions per species: 0.08


## chebi + uniprot

In [10]:
batch_results_df = evaluate_models_in_folder(
    model_dir=model_dir,
    llm_model="Llama-4-Maverick-17B-128E-Instruct-FP8",
    entity_type='auto',
    database=['chebi', 'uniprot'],
    method="direct",
    top_k = 3,
    save_llm_results=True,
    output_dir=output_dir,
    output_file="biomd251106_chebi+uniprot_direct_llama-4_top3_autoType.csv",
    start_at=1
)

2025-12-06 22:37:18,134 - WARNING - Skipping BIOMD0000000001.xml - no results generated


LLM results will be saved to: ./autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_2237
Saved configuration to ./autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_2237/config.txt
Evaluating 1/1075: BIOMD0000000001.xml
Evaluating 2/1075: BIOMD0000000002.xml
LLM results saved to: autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_2237/BIOMD0000000002.txt
Evaluating 3/1075: BIOMD0000000003.xml
LLM results saved to: autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_2237/BIOMD0000000003.txt
Evaluating 4/1075: BIOMD0000000004.xml
LLM results saved to: autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_2237/BIOMD0000000004.txt
Evaluating 5/1075: BIOMD0000000005.xml
LLM results saved to: autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_2237/BIOMD0000000005.txt
Evaluating 6/1075: BIOMD0000000006.xml
LLM results saved to: autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_2237/BIOMD0000000006.txt
Evaluating 7/1075: 

2025-12-06 22:39:42,915 - WARNING - Skipping BIOMD0000000020.xml - no results generated


LLM results saved to: autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_2237/BIOMD0000000019.txt
Evaluating 20/1075: BIOMD0000000020.xml
Evaluating 21/1075: BIOMD0000000021.xml
LLM results saved to: autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_2237/BIOMD0000000021.txt
Evaluating 22/1075: BIOMD0000000022.xml
LLM results saved to: autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_2237/BIOMD0000000022.txt
Evaluating 23/1075: BIOMD0000000023.xml


2025-12-06 22:39:53,391 - WARNING - Skipping BIOMD0000000024.xml - no results generated


...

Evaluating 1071/1075: BIOMD0000001097.xml


2025-12-07 02:28:58,281 - WARNING - Skipping BIOMD0000001097.xml - no results generated
2025-12-07 02:28:58,551 - WARNING - Skipping BIOMD0000001098.xml - no results generated
2025-12-07 02:28:58,889 - WARNING - Skipping BIOMD0000001099.xml - no results generated


Evaluating 1072/1075: BIOMD0000001098.xml
Evaluating 1073/1075: BIOMD0000001099.xml


2025-12-07 02:28:58,906 - WARNING - Skipping BIOMD0000001102.xml - no results generated


Evaluating 1074/1075: BIOMD0000001102.xml
Evaluating 1075/1075: BIOMD0000001103.xml
LLM results saved to: autoType/llama-4-maverick-17b-128e-instruct-fp8/auto/20251206_2237/BIOMD0000001103.txt


In [11]:
print_evaluation_results("autoType/biomd251106_chebi+uniprot_direct_llama-4_top3_autoType.csv", ref_results_csv=None)

Showing all results
Number of models assessed: 683
Number of models with predictions: 617
Number of annotations evaluated: 23765
Average accuracy (per model): 0.28
Ave. recall (formula): 0.25
Ave. precision (formula): 0.23
Ave. recall (exact): 0.24
Ave. precision (exact): 0.14
Average accuracy (per species): 0.44
Ave. recall (formula, per species): 0.42
Ave. precision (formula, per species): 0.40
Ave. recall (exact, per species): 0.28
Ave. precision (exact, per species): 0.28
Ave. total time (per model): 17.26
Ave. total time (per element, per model): 0.50
Ave. LLM time (per model): 16.18
Ave. LLM time (per element, per model): 0.46
Average number of predictions per species: 1.10


# Statistics

In [6]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts_updated.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 312
Number of annotations evaluated: 11373
Average accuracy (per model): 0.86
Ave. recall (formula): 0.86
Ave. precision (formula): 0.65
Ave. recall (exact): 0.73
Ave. precision (exact): 0.26
Average accuracy (per species): 0.93
Ave. recall (formula, per species): 0.93
Ave. precision (formula, per species): 0.69
Ave. recall (exact, per species): 0.60
Ave. precision (exact, per species): 0.39
Ave. total time (per model): 19.55
Ave. total time (per element, per model): 0.55
Ave. LLM time (per model): 18.97
Ave. LLM time (per element, per model): 0.53
Average number of predictions per species: 2.77


In [7]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 308
Number of annotations evaluated: 11370
Average accuracy (per model): 0.85
Ave. recall (formula): 0.85
Ave. precision (formula): 0.64
Ave. recall (exact): 0.72
Ave. precision (exact): 0.25
Average accuracy (per species): 0.91
Ave. recall (formula, per species): 0.91
Ave. precision (formula, per species): 0.67
Ave. recall (exact, per species): 0.59
Ave. precision (exact, per species): 0.37
Ave. total time (per model): 19.25
Ave. total time (per element, per model): 0.54
Ave. LLM time (per model): 18.57
Ave. LLM time (per element, per model): 0.52
Average number of predictions per species: 2.79


In [9]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top10_autoType_updated_prompts_updated.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 313
Number of annotations evaluated: 11373
Average accuracy (per model): 0.89
Ave. recall (formula): 0.89
Ave. precision (formula): 0.23
Ave. recall (exact): 0.84
Ave. precision (exact): 0.09
Average accuracy (per species): 0.95
Ave. recall (formula, per species): 0.95
Ave. precision (formula, per species): 0.26
Ave. recall (exact, per species): 0.68
Ave. precision (exact, per species): 0.15
Ave. total time (per model): 19.59
Ave. total time (per element, per model): 0.55
Ave. LLM time (per model): 18.90
Ave. LLM time (per element, per model): 0.53
Average number of predictions per species: 8.93


In [8]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top10_autoType_updated_prompts.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 306
Number of annotations evaluated: 11370
Average accuracy (per model): 0.87
Ave. recall (formula): 0.87
Ave. precision (formula): 0.23
Ave. recall (exact): 0.82
Ave. precision (exact): 0.09
Average accuracy (per species): 0.93
Ave. recall (formula, per species): 0.93
Ave. precision (formula, per species): 0.25
Ave. recall (exact, per species): 0.67
Ave. precision (exact, per species): 0.14
Ave. total time (per model): 18.79
Ave. total time (per element, per model): 0.53
Ave. LLM time (per model): 18.12
Ave. LLM time (per element, per model): 0.51
Average number of predictions per species: 8.97


In [7]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top10_autoType_updated_prompts.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'], entity_types=['chemical'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Filtering results by entity types: ['chemical']
Number of models assessed: 305
Number of models with predictions: 305
Number of annotations evaluated: 10850
Average accuracy (per model): 0.93
Ave. recall (formula): 0.93
Ave. precision (formula): 0.24
Ave. recall (exact): 0.88
Ave. precision (exact): 0.09
Average accuracy (per species): 0.96
Ave. recall (formula, per species): 0.96
Ave. precision (formula, per species): 0.26
Ave. recall (exact, per species): 0.69
Ave. precision (exact, per species): 0.15
Ave. total time (per model): 19.60
Ave. total time (per element, per model): 0.55
Ave. LLM time (per model): 18.89
Ave. LLM time (per element, per model): 0.53
Average number of predictions per species: 9.28


In [5]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'], entity_types=['chemical'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Filtering results by entity types: ['chemical']
Number of models assessed: 307
Number of models with predictions: 307
Number of annotations evaluated: 10814
Average accuracy (per model): 0.91
Ave. recall (formula): 0.91
Ave. precision (formula): 0.69
Ave. recall (exact): 0.77
Ave. precision (exact): 0.27
Average accuracy (per species): 0.95
Ave. recall (formula, per species): 0.94
Ave. precision (formula, per species): 0.69
Ave. recall (exact, per species): 0.60
Ave. precision (exact, per species): 0.38
Ave. total time (per model): 19.96
Ave. total time (per element, per model): 0.57
Ave. LLM time (per model): 19.26
Ave. LLM time (per element, per model): 0.55
Average number of predictions per species: 2.88


In [4]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated_prompts.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 308
Number of annotations evaluated: 11370
Average accuracy (per model): 0.85
Ave. recall (formula): 0.85
Ave. precision (formula): 0.64
Ave. recall (exact): 0.72
Ave. precision (exact): 0.25
Average accuracy (per species): 0.91
Ave. recall (formula, per species): 0.91
Ave. precision (formula, per species): 0.67
Ave. recall (exact, per species): 0.59
Ave. precision (exact, per species): 0.37
Ave. total time (per model): 19.25
Ave. total time (per element, per model): 0.54
Ave. LLM time (per model): 18.57
Ave. LLM time (per element, per model): 0.52
Average number of predictions per species: 2.79


In [5]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'], entity_types=['chemical'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Filtering results by entity types: ['chemical']
Number of models assessed: 308
Number of models with predictions: 308
Number of annotations evaluated: 10815
Average accuracy (per model): 0.91
Ave. recall (formula): 0.91
Ave. precision (formula): 0.68
Ave. recall (exact): 0.77
Ave. precision (exact): 0.27
Average accuracy (per species): 0.95
Ave. recall (formula, per species): 0.94
Ave. precision (formula, per species): 0.69
Ave. recall (exact, per species): 0.61
Ave. precision (exact, per species): 0.38
Ave. total time (per model): 19.98
Ave. total time (per element, per model): 0.57
Ave. LLM time (per model): 19.28
Ave. LLM time (per element, per model): 0.55
Average number of predictions per species: 2.88


In [14]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top3_autoType.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 316
Number of models with predictions: 310
Number of annotations evaluated: 11154
Average accuracy (per model): 0.88
Ave. recall (formula): 0.88
Ave. precision (formula): 0.66
Ave. recall (exact): 0.75
Ave. precision (exact): 0.26
Average accuracy (per species): 0.94
Ave. recall (formula, per species): 0.94
Ave. precision (formula, per species): 0.69
Ave. recall (exact, per species): 0.61
Ave. precision (exact, per species): 0.38
Ave. total time (per model): 19.10
Ave. total time (per element, per model): 0.54
Ave. LLM time (per model): 18.40
Ave. LLM time (per element, per model): 0.52
Average number of predictions per species: 2.83


In [ ]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-4_top10_autoType.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 315
Number of models with predictions: 309
Number of annotations evaluated: 11145
Average accuracy (per model): 0.91
Ave. recall (formula): 0.91
Ave. precision (formula): 0.24
Ave. recall (exact): 0.85
Ave. precision (exact): 0.09
Average accuracy (per species): 0.95
Ave. recall (formula, per species): 0.95
Ave. precision (formula, per species): 0.25
Ave. recall (exact, per species): 0.69
Ave. precision (exact, per species): 0.14
Ave. total time (per model): 19.39
Ave. total time (per element, per model): 0.55
Ave. LLM time (per model): 18.64
Ave. LLM time (per element, per model): 0.53
Average number of predictions per species: 9.20


In [4]:
print_evaluation_results("autoType/biomd251106_chebi_direct_llama-3_top10_autoType.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 306
Number of models with predictions: 301
Number of annotations evaluated: 10806
Average accuracy (per model): 0.85
Ave. recall (formula): 0.85
Ave. precision (formula): 0.73
Ave. recall (exact): 0.79
Ave. precision (exact): 0.36
Average accuracy (per species): 0.90
Ave. recall (formula, per species): 0.90
Ave. precision (formula, per species): 0.77
Ave. recall (exact, per species): 0.59
Ave. precision (exact, per species): 0.48
Ave. total time (per model): 25.95
Ave. total time (per element, per model): 0.73
Ave. LLM time (per model): 23.66
Ave. LLM time (per element, per model): 0.67
Average number of predictions per species: 2.77


In [6]:
print_evaluation_results("results/biomd251106_chebi_direct_llama-3_top10.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 316
Number of annotations evaluated: 11370
Average accuracy (per model): 0.87
Ave. recall (formula): 0.87
Ave. precision (formula): 0.76
Ave. recall (exact): 0.79
Ave. precision (exact): 0.36
Average accuracy (per species): 0.87
Ave. recall (formula, per species): 0.86
Ave. precision (formula, per species): 0.74
Ave. recall (exact, per species): 0.57
Ave. precision (exact, per species): 0.46
Ave. total time (per model): 25.23
Ave. total time (per element, per model): 0.71
Ave. LLM time (per model): 22.73
Ave. LLM time (per element, per model): 0.64
Average number of predictions per species: 2.67


In [7]:
print_evaluation_results("autoType/biomd251106_chebi_rag_llama-3_top10_autoType.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 307
Number of models with predictions: 307
Number of annotations evaluated: 10826
Average accuracy (per model): 0.94
Ave. recall (formula): 0.94
Ave. precision (formula): 0.23
Ave. recall (exact): 0.88
Ave. precision (exact): 0.09
Average accuracy (per species): 0.95
Ave. recall (formula, per species): 0.94
Ave. precision (formula, per species): 0.22
Ave. recall (exact, per species): 0.68
Ave. precision (exact, per species): 0.13
Ave. total time (per model): 24.71
Ave. total time (per element, per model): 0.70
Ave. LLM time (per model): 23.80
Ave. LLM time (per element, per model): 0.67
Average number of predictions per species: 9.94


In [8]:
print_evaluation_results("results/biomd251106_chebi_rag_llama-3_top10.csv", ref_results_csv=None, bqbiol_qualifiers=['is', 'isVersionOf'])

Showing all results
Filtering results by qualifiers: ['is', 'isVersionOf']
Number of models assessed: 320
Number of models with predictions: 320
Number of annotations evaluated: 11370
Average accuracy (per model): 0.91
Ave. recall (formula): 0.91
Ave. precision (formula): 0.23
Ave. recall (exact): 0.85
Ave. precision (exact): 0.09
Average accuracy (per species): 0.91
Ave. recall (formula, per species): 0.91
Ave. precision (formula, per species): 0.22
Ave. recall (exact, per species): 0.65
Ave. precision (exact, per species): 0.12
Ave. total time (per model): 23.79
Ave. total time (per element, per model): 0.67
Ave. LLM time (per model): 22.84
Ave. LLM time (per element, per model): 0.64
Average number of predictions per species: 9.89


In [8]:
# find different model, species pairs
df1 = pd.read_csv('results/biomd251106_chebi_rag_llama-4_top3.csv')
df2 = pd.read_csv('autoType/biomd251106_chebi_rag_llama-4_top3_autoType_updated.csv')
# only consider is, isVersionOf
df1 = df1[df1['qualifier'].isin(['is', 'isVersionOf'])]
df2 = df2[df2['qualifier'].isin(['is', 'isVersionOf'])]

# find pairs in df1 but not in df2
df1['model_species_pair'] = df1['model'] + '_' + df1['species_id']
df2['model_species_pair'] = df2['model'] + '_' + df2['species_id']
df1[~df1['model_species_pair'].isin(df2['model_species_pair'])].to_csv('autoType/biomd251106_chebi_rag_llama-4_top3_auto_diff.csv', index=False)